%md
# 🥇 Gold Layer — Business Metrics & ML Features

**Project:** Warehouse & Retail Sales ML Pipeline  
**Author:** Santiago López Blanco  
**Date:** May 2026  
**Source:** main.default.silver_warehouse_sales  

## What this notebook produces

| Table | Purpose | Granularity |
|-------|---------|-------------|
| `gold_business_metrics` | KPIs for dashboards and business analysis | year + month + item_type + supplier |
| `gold_ml_features` | Engineered features for demand forecasting | year + month + item_type + supplier_tier |

## Design decisions
- All years included (2017–2020) — gaps documented, not removed
- REF, DUNNAGE, UNCLASSIFIED excluded — not real sales activity
- STR_SUPPLIES and KEGS retained — real categories despite low volume

In [0]:
# ============================================================
#   IMPORTS & SETUP
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("✓ Imports loaded")

%md
## Step 1 — Load Silver Table

Reading from `main.default.silver_warehouse_sales` and applying the same
exclusion filter used throughout the EDA (REF, DUNNAGE, UNCLASSIFIED).

In [0]:
# ============================================================
#   LOAD SILVER TABLE
# ============================================================

df_silver = spark.table("main.default.silver_warehouse_sales")

# Exclude non-commercial categories — same filter as EDA
EXCLUDE = ["REF", "DUNNAGE", "UNCLASSIFIED"]
df = df_silver.filter(~F.col("item_type").isin(EXCLUDE))

# Validate
row_count = df.count()
silver_count = df_silver.count()

print("=" * 50)
print("  SILVER TABLE LOADED")
print("=" * 50)
print(f"  Silver rows     : {silver_count:,}")
print(f"  Clean rows      : {row_count:,}")
print(f"  Excluded rows   : {silver_count - row_count:,}")
print(f"  Item types      : {df.select('item_type').distinct().count()}")
print(f"  Suppliers       : {df.select('supplier').distinct().count()}")
print("=" * 50)

## Step 2 — Build gold_business_metrics

Silver has 307,422 individual transactions. This step collapses them into
one row per **year + month + item_type + supplier** combination — a single
row that summarizes everything that happened in that group.

**Example:**
> Crown Imports sold 865 units of BEER in January 2019 across 4 transactions,
> 85% of which went through the warehouse channel.

**Metrics computed:**

| Column | What it means |
|--------|--------------|
| `total_sales` | Total units moved by this group in this month |
| `retail_sales` | Units sold through retail stores |
| `retail_transfers` | Units moved between stores (not new sales) |
| `warehouse_sales` | Units sold through warehouse (B2B) |
| `transaction_count` | How many individual orders this group made |
| `avg_per_transaction` | Average units per order |
| `market_share_pct` | % of total historical market volume |
| `warehouse_ratio` | Warehouse share of total (0 = all retail, 1 = all warehouse) |

In [0]:
# ============================================================
#   STEP 2 — BUILD gold_business_metrics
# ============================================================

# Calculate total market volume once — used for market_share_pct
total_market = df.agg(F.sum("total_sales")).collect()[0][0]

business_metrics = df \
    .groupBy("year", "month", "item_type", "supplier") \
    .agg(
        F.round(F.sum("total_sales"), 2).alias("total_sales"),
        F.round(F.sum("retail_sales"), 2).alias("retail_sales"),
        F.round(F.sum("retail_transfers"), 2).alias("retail_transfers"),
        F.round(F.sum("warehouse_sales"), 2).alias("warehouse_sales"),
        F.count("*").alias("transaction_count"),
        F.round(F.avg("total_sales"), 2).alias("avg_per_transaction")
    ) \
    .withColumn(
        "market_share_pct",
        F.round(F.col("total_sales") / total_market * 100, 4)
    ) \
    .withColumn(
        "warehouse_ratio",
        F.round(
            F.when(F.col("total_sales") == 0, 0)
             .otherwise(F.col("warehouse_sales") / F.col("total_sales")),
            4
        )
    ) \
    .orderBy("year", "month", F.col("total_sales").desc())

row_count = business_metrics.count()
print(f"✓ gold_business_metrics built — {row_count:,} rows")

display(business_metrics)

%md
## Step 3 — Save gold_business_metrics as Delta Table

Writing the aggregated business metrics to the catalog as a permanent Delta Table.
Once saved, this table can be queried directly from SQL Editor or connected to a dashboard.

In [0]:
# ============================================================
#   STEP 3 — SAVE gold_business_metrics
# ============================================================

business_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.gold_business_metrics")

# Validate save
saved_count = spark.table("main.default.gold_business_metrics").count()

print("=" * 50)
print("  GOLD BUSINESS METRICS — SAVED")
print("=" * 50)
print(f"  Table   : main.default.gold_business_metrics")
print(f"  Rows    : {saved_count:,}")
print(f"  Columns : 12")
print(f"  Status  : ✓ Complete")
print("=" * 50)

## Step 4 — Build gold_ml_features

This table transforms the business metrics into features ready for a demand
forecasting model. The goal is to predict `next_month_sales` for a given
`year + month + item_type + supplier` combination.

Only suppliers with **at least 6 months of history** are included — groups
with fewer months can't produce valid lag features.

**Result:** 503 active groups · 8,219 rows · avg 16.3 rows per group

| Feature | Type | Description |
|---------|------|-------------|
| `lag_1_sales` | Numeric | Sales from previous month — strongest single predictor |
| `lag_2_sales` | Numeric | Sales from 2 months ago — captures short-term trend |
| `rolling_3m_avg` | Numeric | Avg of 3 previous months — smooths one-off spikes, no leakage |
| `warehouse_ratio` | Numeric | Warehouse share of total sales (0 to 1) |
| `supplier_tier` | Categorical | top3 / top15 / rest — groups 396 suppliers into 3 tiers |
| `month_sin` / `month_cos` | Numeric | Seasonality as a cycle — Dec and Jan are neighbors, not opposites |
| `next_month_sales` | **Target** | What the model will learn to predict |

> **Train / Test split:** 2017–2019 → train (7,101 rows) · 2020 → test (1,118 rows)

In [0]:
# ============================================================
#   STEP 4 — BUILD gold_ml_features (rebuilt)
#   Granularity: year + month + item_type + supplier
#   Filter: minimum 6 months of history per group
# ============================================================

# ── 1. Aggregate by year + month + item_type + supplier ──────
df_agg = business_metrics \
    .groupBy("year", "month", "item_type", "supplier") \
    .agg(
        F.round(F.sum("total_sales"), 2).alias("total_sales"),
        F.round(F.avg("warehouse_ratio"), 4).alias("warehouse_ratio"),
        F.sum("transaction_count").alias("transaction_count")
    ) \
    .orderBy("item_type", "supplier", "year", "month")

# ── 2. Keep only groups with at least 6 months of history ────
# Count how many months each item_type + supplier combination has
months_per_group = df_agg \
    .groupBy("item_type", "supplier") \
    .agg(F.count("*").alias("months_available"))

# Filter to groups with >= 6 months
active_groups = months_per_group.filter(F.col("months_available") >= 6)

# Join back to keep only those groups
df_filtered = df_agg.join(
    active_groups.select("item_type", "supplier"),
    on=["item_type", "supplier"],
    how="inner"
)

# ── 3. Add supplier_tier ─────────────────────────────────────
TOP_3_SUPPLIERS = ["CROWN IMPORTS", "ANHEUSER BUSCH INC", "MILLER BREWING COMPANY"]
TOP_15_SUPPLIERS = [
    "CROWN IMPORTS", "ANHEUSER BUSCH INC", "MILLER BREWING COMPANY",
    "HEINEKEN USA", "E & J GALLO WINERY", "DIAGEO NORTH AMERICA INC",
    "CONSTELLATION BRANDS", "BOSTON BEER CORPORATION", "THE WINE GROUP",
    "JIM BEAM BRANDS CO", "FLYING DOG BREWERY LLLP", "YUENGLING BREWERY",
    "SAZERAC CO", "BACARDI USA INC", "PABST BREWING CO"
]

df_tiered = df_filtered.withColumn(
    "supplier_tier",
    F.when(F.col("supplier").isin(TOP_3_SUPPLIERS), "top3")
     .when(F.col("supplier").isin(TOP_15_SUPPLIERS), "top15")
     .otherwise("rest")
)

# ── 4. Define window per item_type + supplier ────────────────
w = Window.partitionBy("item_type", "supplier").orderBy("year", "month")

# ── 5. Engineer lag and rolling features ─────────────────────
df_features = df_tiered \
    .withColumn("lag_1_sales", F.lag("total_sales", 1).over(w)) \
    .withColumn("lag_2_sales", F.lag("total_sales", 2).over(w)) \
    .withColumn(
        "rolling_3m_avg",
        F.round(
            F.avg("total_sales").over(w.rowsBetween(-3, -1)),
            2
        )
    ) \
    .withColumn(
        "next_month_sales",
        F.lead("total_sales", 1).over(w)
    ) \
    .withColumn(
        "month_sin",
        F.round(F.sin(F.col("month") * F.lit(2 * 3.141592653589793 / 12)), 6)
    ) \
    .withColumn(
        "month_cos",
        F.round(F.cos(F.col("month") * F.lit(2 * 3.141592653589793 / 12)), 6)
    )

# ── 6. Drop rows where target or lags are null ───────────────
df_ml = df_features.filter(
    F.col("next_month_sales").isNotNull() &
    F.col("lag_1_sales").isNotNull() &
    F.col("lag_2_sales").isNotNull() &
    F.col("rolling_3m_avg").isNotNull()
)

row_count = df_ml.count()
group_count = df_ml.select("item_type", "supplier").distinct().count()

print("=" * 55)
print("  GOLD ML FEATURES")
print("=" * 55)
print(f"  Active groups   : {group_count:,} (item_type + supplier)")
print(f"  Total rows      : {row_count:,}")
print(f"  Avg rows/group  : {round(row_count/group_count, 1)}")
print("=" * 55)

display(df_ml)

%md
## Step 5 — Save gold_ml_features as Delta Table

Writing the ML-ready feature table to the catalog.
This table is the direct input for the forecasting model in the next notebook.

In [0]:
# ============================================================
#   STEP 5 — SAVE gold_ml_features 
# ============================================================

df_ml.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.gold_ml_features")

saved_count = spark.table("main.default.gold_ml_features").count()

print("=" * 55)
print("  GOLD ML FEATURES — SAVED")
print("=" * 55)
print(f"  Table     : main.default.gold_ml_features")
print(f"  Rows      : {saved_count:,}")
print(f"  Columns   : 14")
print(f"  Target    : next_month_sales")
print(f"  Status    : ✓ Complete")
print("=" * 55)

---

## Gold Layer — Summary

### Tables Created

| Table | Rows | Columns | Purpose |
|-------|------|---------|---------|
| `gold_business_metrics` | 9,980 | 12 | KPIs for business analysis and dashboards |
| `gold_ml_features` | 8,219 | 14 | Feature table for demand forecasting model |

### What was built

**gold_business_metrics**
- Aggregated Silver (307,422 rows) into one row per year + month + item_type + supplier
- Added `market_share_pct` and `warehouse_ratio` for business context
- Covers 396 suppliers across 6 item types and 24 months

**gold_ml_features**
- Granularity: year + month + item_type + supplier (503 active groups)
- Filter: minimum 6 months of history per group — ensures valid lag features
- Engineered lag features: `lag_1_sales`, `lag_2_sales`, `rolling_3m_avg` (no leakage)
- Added seasonality encoding: `month_sin`, `month_cos`
- Added `supplier_tier`: top3 / top15 / rest
- Target variable: `next_month_sales`
- Train set (2017–2019): 7,101 rows — Test set (2020): 1,118 rows

### Next Step
`05_ml_model` — Train a demand forecasting model using `gold_ml_features`.
Predict `next_month_sales` from lag features, seasonality, item type, and supplier tier.

---
*Notebook: `04_gold_layer` — Santiago López Blanco — May 2026*